In [ ]:
# Stanford Heart Transplant Data Analysis (Python)
# This notebook demonstrates:
# 1. Loading preprocessed data from cache
# 2. Creating Table One (baseline characteristics)

import sys
from pathlib import Path

_examples = Path.cwd().resolve().parent
if (_examples / "support.py").exists() and str(_examples) not in sys.path:
    sys.path.insert(0, str(_examples))

import config
import os
import polars as pl
from support import load_data

# キャッシュからデータフレームを読み込み
all_dfs = load_data(config.CACHE / "build")
target = all_dfs["target"]

print(f"Loaded {len(target)} records")
print(f"Columns: {target.columns}")



In [ ]:
# ログディレクトリの取得(Taskfileから実行された場合はLOGDIR環境変数が設定される)
logdir = Path(os.environ.get("LOGDIR", "log/default"))
logdir.mkdir(parents=True, exist_ok=True)

print(f"Log directory: {logdir}")



In [ ]:
# Table One: Baseline characteristics
# 移植群と非移植群で比較

from statract import tableone, agg_category, agg_mean_sd

# 各患者の最初の行のみを使用（baseline characteristics）
# idでグループ化して最初の行を取得
df_baseline = target.group_by("id").first()

# Table Oneパラメータの定義
params = {
    "N": (pl.col("id"), "count", None),
    "Age (normalized)": (pl.col("age_normalized"), agg_mean_sd, None),
    "Year": (pl.col("year"), agg_mean_sd, None),
    "Surgery, n (%)": (pl.col("surgery"), agg_category, None),
    "Transplant, n (%)": (pl.col("transplant"), agg_category, None),
}

# transplantで層別化してTable Oneを作成
table1 = tableone(
    df_baseline,
    params,
    hue="transplant",
    add_all=True,
    add_pvalue=True,
)

print("Table One: Baseline Characteristics")
print(table1)

# CSVとして保存
table1.write_csv(logdir / "table1_baseline.csv")

